### Preprocessing

- Convert text to lowercase
- Tokenization
- Remove punctuation
- Remove stopwords (and, a, the)
- Lemmatization

In [1]:
# Setup
from pathlib import Path
import re

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(resource, quiet=True)

csv_path = Path('data/amazon_reviews.csv')
if not csv_path.exists():
    raise FileNotFoundError(f"CSV file not found: {csv_path}")

df = pd.read_csv(csv_path)
if 'Review Description' not in df.columns:
    raise KeyError("Expected column 'Review Description' in the dataset.")

df.head(3)


,Star Rating,Review Title,Review Description
0,5.0 out of 5 stars,Good Phone for long battery usage,Good phoneGood battery lifeDecent cameraVery h...
1,5.0 out of 5 stars,Good phone for studying longer,"Bought this phone for my brother .Nice phone ,..."
2,5.0 out of 5 stars,Very Good Phone,"Very Nice Phone, as I expected, I am using 2 p..."


In [2]:
# Convert text to lowercase
df['review_base'] = (
    df['Review Description']
    .fillna('')
    .astype(str)
    .str.replace(r'\s*Read more\s*$', '', regex=True, flags=re.IGNORECASE)
    .str.replace(r'(?<=[a-z])(?=[A-Z])', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

df = df[df['review_base'].ne('')].copy()

df['review_lower'] = df['review_base'].str.lower()

df[['Review Description', 'review_base', 'review_lower']].head(3)


,Review Description,review_base,review_lower
0,Good phoneGood battery lifeDecent cameraVery h...,Good phone Good battery life Decent camera Ver...,good phone good battery life decent camera ver...
1,"Bought this phone for my brother .Nice phone ,...","Bought this phone for my brother .Nice phone ,...","bought this phone for my brother .nice phone ,..."
2,"Very Nice Phone, as I expected, I am using 2 p...","Very Nice Phone, as I expected, I am using 2 p...","very nice phone, as i expected, i am using 2 p..."


In [3]:
# Tokenization
df['review_tokens'] = df['review_lower'].apply(word_tokenize)

df[['review_lower', 'review_tokens']].head(3)


,review_lower,review_tokens
0,good phone good battery life decent camera ver...,"[good, phone, good, battery, life, decent, cam..."
1,"bought this phone for my brother .nice phone ,...","[bought, this, phone, for, my, brother, .nice,..."
2,"very nice phone, as i expected, i am using 2 p...","[very, nice, phone, ,, as, i, expected, ,, i, ..."


In [4]:
# Remove punctuation
df['review_tokens_no_punct'] = df['review_tokens'].apply(
    lambda tokens: [token for token in tokens if token.isalnum()]
)

df[['review_tokens', 'review_tokens_no_punct']].head(3)


,review_tokens,review_tokens_no_punct
0,"[good, phone, good, battery, life, decent, cam...","[good, phone, good, battery, life, decent, cam..."
1,"[bought, this, phone, for, my, brother, .nice,...","[bought, this, phone, for, my, brother, phone,..."
2,"[very, nice, phone, ,, as, i, expected, ,, i, ...","[very, nice, phone, as, i, expected, i, am, us..."


In [5]:
# Remove stopwords
stop_words = set(stopwords.words('english'))
df['review_tokens_no_stopwords'] = df['review_tokens_no_punct'].apply(
    lambda tokens: [token for token in tokens if token not in stop_words]
)

df[['review_tokens_no_punct', 'review_tokens_no_stopwords']].head(3)


,review_tokens_no_punct,review_tokens_no_stopwords
0,"[good, phone, good, battery, life, decent, cam...","[good, phone, good, battery, life, decent, cam..."
1,"[bought, this, phone, for, my, brother, phone,...","[bought, phone, brother, phone, suitable, long..."
2,"[very, nice, phone, as, i, expected, i, am, us...","[nice, phone, expected, using, 2, phones, iqoo..."


In [6]:
# Lemmatization
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    cleaned_tokens = []
    for token in tokens:
        if token.isalpha() and len(token) > 2:
            cleaned_tokens.append(lemmatizer.lemmatize(token))
        else:
            cleaned_tokens.append(token)
    return cleaned_tokens

df['review_tokens_lemmatized'] = df['review_tokens_no_stopwords'].apply(lemmatize_tokens)

def build_final_text(row):
    if row['review_tokens_lemmatized']:
        return ' '.join(row['review_tokens_lemmatized'])
    if row['review_tokens_no_punct']:
        return ' '.join(row['review_tokens_no_punct'])
    return row['review_lower']

df['review_text_final'] = df.apply(build_final_text, axis=1)

df[['review_base', 'review_text_final']].head(3)


,review_base,review_text_final
0,Good phone Good battery life Decent camera Ver...,good phone good battery life decent camera hap...
1,"Bought this phone for my brother .Nice phone ,...",bought phone brother phone suitable long hour ...
2,"Very Nice Phone, as I expected, I am using 2 p...",nice phone expected using 2 phone iqoo z10x z1...


In [7]:
# Export preprocessed dataset
export_columns = ['Star Rating', 'Review Title', 'Review Description', 'review_text_final']
export_df = df[export_columns].copy()

export_path = Path('data/amazon_reviews_preprocessed.csv')
export_df.to_csv(export_path, index=False, encoding='utf-8')

print(f"Saved preprocessed dataset to: {export_path}")
print(f"Dataset shape: {export_df.shape}")
export_df.head(3)


Saved preprocessed dataset to: data/amazon_reviews_preprocessed.csv
Dataset shape: (688, 4)


,Star Rating,Review Title,Review Description,review_text_final
0,5.0 out of 5 stars,Good Phone for long battery usage,Good phoneGood battery lifeDecent cameraVery h...,good phone good battery life decent camera hap...
1,5.0 out of 5 stars,Good phone for studying longer,"Bought this phone for my brother .Nice phone ,...",bought phone brother phone suitable long hour ...
2,5.0 out of 5 stars,Very Good Phone,"Very Nice Phone, as I expected, I am using 2 p...",nice phone expected using 2 phone iqoo z10x z1...
